In [58]:
# Import python's built-in regular expression library
import re
import os
from pathlib import Path
import anthropic

def load_env_file(env_path=".env"):
    env_path = Path(env_path)

    if not env_path.exists():
        raise FileNotFoundError(f"Could not find .env file at: {env_path.resolve()}")

    with env_path.open("r") as file:
        for line in file:
            line = line.strip()

            if not line or line.startswith("#"):
                continue

            if "=" not in line:
                continue

            key, value = line.split("=", 1)
            os.environ[key.strip()] = value.strip().strip('"').strip("'")

load_env_file(".env")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
MODEL_NAME = os.environ.get("MODEL_NAME")

if not API_KEY:
    raise ValueError("Missing ANTHROPIC_API_KEY in .env")

if not MODEL_NAME:
    raise ValueError("Missing MODEL_NAME in .env")

client = anthropic.Anthropic(api_key=API_KEY)

print("Loaded API key and model name from .env")


Loaded API key and model name from .env


In [59]:
# writing helper functions to keep track of message history - otherwise Claude would not have the message history

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None):
    params = {
        "model": MODEL_NAME,
        "max_tokens": 1000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [60]:
# creating a chat history to capture to have a longer exchange

# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "Define quantum computing in one sentence")

# Get Claude's response
answer = chat(messages)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

# Add a follow-up question
add_user_message(messages, "Write another sentence")

# Get the follow-up response with full context
final_answer = chat(messages)

In [61]:
# printing the answer and final answer for review
print(answer)
print(final_answer)

Quantum computing harnesses the properties of quantum mechanics—such as superposition and entanglement—to process information in fundamentally different ways than classical computers, enabling certain calculations to be performed exponentially faster.
Unlike classical bits that exist as either 0 or 1, quantum bits (qubits) can exist in a superposition of both states simultaneously, allowing quantum computers to explore multiple solutions in parallel.


In [62]:
# excerciese for system prompts

messages = []
add_user_message(messages,
                 "Write a Python function that checks a string for dublicate characters.",
                 )
answer = chat(messages, system="You are a Python engineer who write very concise code")

answer

'# Check for duplicate characters\n\n```python\ndef has_duplicates(s: str) -> bool:\n    return len(s) != len(set(s))\n```\n\n**Usage:**\n```python\nprint(has_duplicates("hello"))      # True (l appears twice)\nprint(has_duplicates("python"))     # False\nprint(has_duplicates("aab"))        # True\n```\n\n**Alternative (case-insensitive):**\n```python\ndef has_duplicates(s: str) -> bool:\n    s = s.lower()\n    return len(s) != len(set(s))\n```'

In [76]:
# updated chat function to include temparature

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": MODEL_NAME,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,

    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [64]:
# playing around with temparature setting

messages = []
add_user_message(messages,
                 "Generate a movie title for a new Marvel summer blockbuster.",
                 )
answer = chat(messages, temperature=1.0,  system="You are a Joe Rogan.")

answer

'# Avengers: The Quantum Fracture\n\nPicture this: A dimensional rift opens in the heart of New York, and fragments of alternate realities start bleeding into our world. The Avengers have to team up with some unexpected allies—including heroes from those fractured dimensions—to stop a cosmic entity that feeds on collapsing timelines.\n\nYou got your big action set pieces, your multiverse stuff that people are eating up right now, and room for character drama. Plus, "Quantum Fracture" just *sounds* like a summer blockbuster, man. It\'s got that perfect balance of sci-fi and catastrophe that makes people want to drop $18 on a ticket and a large popcorn.'

In [65]:
# basic streaming implementation

messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model= MODEL_NAME,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_011Cd95v1Y9YQDcGXXq5wXWz', container=None, content=[], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=18, output_tokens=1, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='#', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' FakeDB\n\nA lightweight in-memory database that generates realistic but entirely fictional data across', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=Te

In [66]:
# introducing simplified text streaming

with client.messages.stream(
    model=MODEL_NAME,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")

# Fake Database Description

The "CloudMirror" database is a distributed, in-memory storage system that uses quantum-inspired hashing algorithms to achieve sub-nanosecond query responses while automatically synchronizing across multiple dimensional nodes for real-time data consistency.

In [67]:
# simplifed streaming and a final message for storage

with client.messages.stream(
    model=MODEL_NAME,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        # Send each chunk to your client
        pass

    # Get the complete message for database storage
    final_message = stream.get_final_message()

In [80]:
# getting claude to write structured files, using JSON as an example

messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])

In [82]:
# cleanig up the json file
import json

# Clean up and parse the JSON
clean_json = json.loads(text.strip())

clean_json

{'Name': 'MyRule',
 'EventBusName': 'default',
 'EventPattern': {'source': ['aws.ec2'],
  'detail-type': ['EC2 Instance State-change Notification'],
  'detail': {'state': ['running']}},
 'State': 'ENABLED',
 'Targets': [{'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:MyFunction',
   'Id': '1'}]}

In [85]:
# structured data excercise

messages = []

prompt = """ Generate three different sample AWS CLI commands. Each should be very short"""

add_user_message(messages, prompt)
add_assistant_message(messages, "Here are all three commands in a single block without any comments:\n```bash")

text = chat(messages, stop_sequences=["```"])

text.strip()

'aws s3 ls\naws ec2 describe-instances --region us-east-1\naws dynamodb list-tables'

In [86]:
from IPython.display import Markdown

Markdown(text)


aws s3 ls
aws ec2 describe-instances --region us-east-1
aws dynamodb list-tables
